# Ablation 4 Smoke Test - DDI Hard Negatives

**Purpose**: Check DDI dataset loading and training initialization before the full run

**Tested**:
1. DDI train/test split from metadata (stratified)
2. Mixed dataset creation (wounds + DDI)
3. DDI-only dataset creation
4. Strategy 4a initialization (1 epoch)
5. Strategy 4b/4c checkpoint loading

**Runtime**: ~5-10 minutes on CPU


In [ ]:
# Lock NumPy to prevent ultralytics from upgrading to 2.x
import numpy as np
import subprocess

default_numpy = np.__version__
print(f"Installing ultralytics...")
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print(f"Locking NumPy to {default_numpy}...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

import cv2
print(f"OK NumPy: {np.__version__}")
print(f"OK OpenCV: {cv2.__version__}")


In [ ]:
import os
import shutil
import time
from pathlib import Path
import pandas as pd
from ultralytics import YOLO

# Paths
BASE_INPUT_PATH = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working")
WOUND_SRC = BASE_INPUT_PATH / "baseline-outputs"


In [ ]:
# TEST 1: DDI Dataset Split from Metadata
print("="*80)
print("TEST 1: DDI SPLIT FROM METADATA")
print("="*80)

DDI_DATASET_PATH = BASE_INPUT_PATH / "ddi-diverse-dermatology"

# Validate dataset structure first
print("\nValidating DDI dataset structure...")
csv_files = list(DDI_DATASET_PATH.glob("*.csv"))
print(f"CSV files found: {[f.name for f in csv_files]}")

# Check for metadata
ddi_metadata_path = DDI_DATASET_PATH / "ddi_metadata_cleaned.csv"
assert ddi_metadata_path.exists(), f"Missing: ddi_metadata_cleaned.csv"
print(f"OK Found: {ddi_metadata_path.name}")

# Load and split
ddi_df = pd.read_csv(ddi_metadata_path)
print(f"\nTotal DDI images: {len(ddi_df)}")
print(f"Columns: {ddi_df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(ddi_df.head(3))

# 80/20 stratified split
from sklearn.model_selection import train_test_split
ddi_train_df, ddi_test_df = train_test_split(
    ddi_df,
    test_size=0.2,
    random_state=42,
    stratify=ddi_df['malignant']
)

print(f"\nOK Split successful:")
print(f"  Train: {len(ddi_train_df)} images")
print(f"  Test: {len(ddi_test_df)} images")

# Find image directory (check all subdirectories)
print(f"\nSearching for image directory...")
subdirs = [d for d in DDI_DATASET_PATH.iterdir() if d.is_dir()]
print(f"Available directories: {[d.name for d in subdirs]}")

# Find directory with PNG images
image_dirs = []
for d in subdirs:
    png_count = len(list(d.glob("*.png")))
    if png_count > 0:
        image_dirs.append((d, png_count))
        print(f"  {d.name}: {png_count} PNG files")

assert len(image_dirs) > 0, f"No PNG images found in {DDI_DATASET_PATH}"
ddi_images_src = image_dirs[0][0]  # Use first directory with PNGs
sample_images = list(ddi_images_src.glob("*.png"))[:3]
print(f"\nOK Using image directory: {ddi_images_src.name}")
print(f"  Total images: {image_dirs[0][1]}")
print(f"  Sample: {[img.name for img in sample_images]}")


In [ ]:
# TEST 2: Mixed Dataset Creation (2 DDI + 2 wounds)
print("\n" + "="*80)
print("TEST 2: MIXED DATASET CREATION (MINI)")
print("="*80)

MIXED_DATASET = OUTPUT_PATH / "mixed_dataset_mini"
(MIXED_DATASET / "train" / "images").mkdir(parents=True, exist_ok=True)
(MIXED_DATASET / "train" / "labels").mkdir(parents=True, exist_ok=True)
(MIXED_DATASET / "test" / "images").mkdir(parents=True, exist_ok=True)
(MIXED_DATASET / "test" / "labels").mkdir(parents=True, exist_ok=True)

# Copy 2 wound images
wound_train_imgs = list((WOUND_SRC / "train_images").glob("*.png"))[:2]
print(f"\nCopying {len(wound_train_imgs)} wound images...")
for img in wound_train_imgs:
    shutil.copy2(img, MIXED_DATASET / "train" / "images" / img.name)

    # Copy corresponding label
    label_src = WOUND_SRC / "yolo_labels" / "train" / f"{img.stem}.txt"
    if label_src.exists():
        shutil.copy2(label_src, MIXED_DATASET / "train" / "labels" / f"{img.stem}.txt")

# Copy 2 DDI images (hard negatives)
ddi_copied = 0
for idx, row in ddi_train_df.head(2).iterrows():
    img_filename = row['DDI_file']
    src_img = ddi_images_src / img_filename

    if src_img.exists():
        dst_img = MIXED_DATASET / "train" / "images" / f"ddi_{img_filename}"
        shutil.copy2(src_img, dst_img)

        # Empty label (no wound)
        label_file = MIXED_DATASET / "train" / "labels" / f"ddi_{src_img.stem}.txt"
        label_file.touch()
        ddi_copied += 1
    else:
        print(f"  WARNING Missing: {img_filename}")

print(f"\nOK Mixed dataset created:")
print(f"  Wound images: {len(wound_train_imgs)}")
print(f"  DDI images: {ddi_copied}")
print(f"  Total train: {len(list((MIXED_DATASET / 'train' / 'images').glob('*')))} images")

# Create YAML config
from pathlib import Path
mixed_config = OUTPUT_PATH / "mixed_dataset_mini.yaml"
mixed_config.write_text(f"""
path: {MIXED_DATASET}
train: train/images
val: train/images
test: train/images

names:
  0: wound
""")
print(f"\nOK Config: {mixed_config}")


In [ ]:
# TEST 3: DDI-Only Dataset Creation (3 images)
print("\n" + "="*80)
print("TEST 3: DDI-ONLY DATASET (MINI)")
print("="*80)

DDI_ONLY_DATASET = OUTPUT_PATH / "ddi_only_mini"
(DDI_ONLY_DATASET / "train" / "images").mkdir(parents=True, exist_ok=True)
(DDI_ONLY_DATASET / "train" / "labels").mkdir(parents=True, exist_ok=True)

# Copy 3 DDI images
ddi_only_copied = 0
for idx, row in ddi_train_df.head(3).iterrows():
    img_filename = row['DDI_file']
    src_img = ddi_images_src / img_filename

    if src_img.exists():
        dst_img = DDI_ONLY_DATASET / "train" / "images" / img_filename
        shutil.copy2(src_img, dst_img)

        # Empty label
        label_file = DDI_ONLY_DATASET / "train" / "labels" / f"{src_img.stem}.txt"
        label_file.touch()
        ddi_only_copied += 1

print(f"\nOK DDI-only dataset created:")
print(f"  Images: {ddi_only_copied}")

# Create YAML
ddi_config = OUTPUT_PATH / "ddi_only_mini.yaml"
ddi_config.write_text(f"""
path: {DDI_ONLY_DATASET}
train: train/images
val: train/images
test: train/images

names:
  0: wound
""")
print(f"OK Config: {ddi_config}")


In [ ]:
# TEST 4: Strategy 4a - Mixed Training (1 epoch)
print("\n" + "="*80)
print("TEST 4: STRATEGY 4a INITIALIZATION")
print("="*80)

# Create fresh model
model_4a = YOLO("yolov8n-seg.pt")

print("\nStarting 1 epoch training on mixed dataset...")
start = time.time()

try:
    results_4a = model_4a.train(
        data=str(mixed_config),
        epochs=1,  # Just 1 epoch for smoke test
        batch=2,   # Tiny batch
        imgsz=128, # Small images for speed
        workers=2,
        verbose=True,
        project=str(OUTPUT_PATH / "smoke_test_4a"),
        name="test",
        exist_ok=True
    )
    elapsed = time.time() - start
    print(f"\nOK Strategy 4a initialized successfully ({elapsed:.1f}s)")
    print(f"OK Model trained on mixed dataset (wounds + DDI)")

except Exception as e:
    print(f"\nFAIL Strategy 4a FAILED: {e}")
    raise


In [ ]:
# TEST 5: Strategy 4b/4c - Checkpoint Loading
print("\n" + "="*80)
print("TEST 5: STRATEGIES 4b/4c CHECKPOINT LOADING")
print("="*80)

# Find the trained checkpoint from 4a
best_4a = OUTPUT_PATH / "smoke_test_4a" / "test" / "weights" / "best.pt"
assert best_4a.exists(), f"Missing 4a checkpoint: {best_4a}"
print(f"\nOK Found 4a checkpoint: {best_4a}")

# Test loading checkpoint
print("\nTesting checkpoint load for fine-tuning...")
try:
    model_4b = YOLO(str(best_4a))
    print(f"OK Strategy 4b: Checkpoint loaded successfully")

    # Verify we can initialize training on DDI-only
    print("\nInitializing DDI-only fine-tuning (1 batch)...")
    results_4b = model_4b.train(
        data=str(ddi_config),
        epochs=1,
        batch=2,
        imgsz=128,
        workers=2,
        verbose=True,
        project=str(OUTPUT_PATH / "smoke_test_4b"),
        name="test",
        exist_ok=True
    )
    print(f"OK Strategy 4b: Fine-tuning initialized successfully")

except Exception as e:
    print(f"FAIL Strategy 4b/4c FAILED: {e}")
    raise


In [ ]:
# SMOKE TEST SUMMARY
print("\n" + "="*80)
print("SMOKE TEST COMPLETE - ALL TESTS PASSED OK")
print("="*80)

print("""
OK Test 1: DDI split from metadata - PASSED
OK Test 2: Mixed dataset creation - PASSED
OK Test 3: DDI-only dataset creation - PASSED
OK Test 4: Strategy 4a initialization - PASSED
OK Test 5: Strategy 4b/4c checkpoint loading - PASSED

READY FOR PRODUCTION DEPLOYMENT (v9)
Expected runtime: 2-3 hours on P100 GPU
""")

# Save completion marker
import json
completion = {
    "status": "PASSED",
    "tests_run": 5,
    "tests_passed": 5,
    "ddi_split_train": len(ddi_train_df),
    "ddi_split_test": len(ddi_test_df),
    "ready_for_production": True
}
with open(OUTPUT_PATH / "smoke_test_completion.json", 'w') as f:
    json.dump(completion, f, indent=2)

print(f"\nOK Completion marker: {OUTPUT_PATH}/smoke_test_completion.json")
